# Notebook D: Q&A Dataset Preparation

**Goal**: Load the synthetic Spurgeon Q&A dataset (`spurgeon_train_1500.jsonl`), parse it into a Hugging Face `Dataset` structure, apply the Qwen2.5 ChatML template, split it into train/val sets (95/5), and save it to disk for training.

## 1. Environment Detection & Setup

In [1]:
import os
from pathlib import Path

# Detect environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IS_COLAB = "COLAB_GPU" in os.environ

if IS_KAGGLE:
    print("Running on Kaggle")
    INPUT_DATASET_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset/spurgeon_qa_train_final.jsonl"
    OUTPUT_DIR = "/kaggle/working/"
elif IS_COLAB:
    print("Running on Google Colab")
    # Expect file to be uploaded or loaded from drive
    INPUT_DATASET_PATH = "/content/spurgeon_train.jsonl"
    OUTPUT_DIR = "/content/"
else:
    print("Running locally")
    INPUT_DATASET_PATH = "../data/spurgeon_train.jsonl"
    OUTPUT_DIR = "../data/"

print(f"Input path: {INPUT_DATASET_PATH}")
print(f"Output directory: {OUTPUT_DIR}")

Running on Kaggle
Input path: /kaggle/input/datasets/rafaelvieira1/spurgeon-qa-dataset/spurgeon_qa_train_final.jsonl
Output directory: /kaggle/working/


## 2. Install Dependencies (Kaggle/Colab only)

In [2]:
if IS_KAGGLE or IS_COLAB:
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
    !pip install -q --no-deps xformers "trl" peft accelerate bitsandbytes datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.4/924.4 kB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 89.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 69.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Load JSONL Dataset

In [3]:
import json
from datasets import Dataset

records = []
if not Path(INPUT_DATASET_PATH).exists():
    raise FileNotFoundError(f"Dataset not found at {INPUT_DATASET_PATH}. Please upload it first.")

with open(INPUT_DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

print(f"Loaded {len(records):,} total training examples.")

# Load as HF Dataset
dataset = Dataset.from_list(records)
print(dataset)

Loaded 2,787 total training examples.
Dataset({
    features: ['messages'],
    num_rows: 2787
})


## 4. Tokenize & Format with ChatML Template

In [ ]:
from transformers import AutoTokenizer
from unsloth.chat_templates import get_chat_template

# Load tokenizer only (we just need the tokenizer to format the templates)
model_name = "unsloth/Qwen2.5-3B-Instruct" if not IS_KAGGLE else "/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    fix_mistral_regex=True,
)

# Apply ChatML template for Qwen2.5 (required since base models lack default templates)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def format_chatML(example):
    messages = example["messages"]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted}

# Apply the mapping
dataset = dataset.map(format_chatML)
print("Sample formatted example:")
print(dataset[0]["text"][:800])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.1: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

The tokenizer you are loading from '/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/kaggle/input/datasets/rafaelvieira1/spurgeon-lora-final/spurgeon_f16_gguf' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/2787 [00:00<?, ? examples/s]

Sample formatted example:
<|im_start|>system
You are Charles Haddon Spurgeon. Answer using only the information in the provided CONTEXT. Stay very close to the actual text.<|im_end|>
<|im_start|>user
CONTEXT:
a grand day for the kingdom of Christ when the King is proclaimed in the streets, when the great trumpet is sounded, when the disciples stand in the highways, and the voice of wisdom is lifted up in the chief places of concourse, at the going in of the gates. Then are things well ordered when Zion lifts up her voice, yea, lifts it up with strength, and saith unto the cities of Judah, 'Behold your God.' Our commission as preachers is to every creature, and, therefore, the more public the teaching of the gospel the better. Truly, there was grace in the earth when in popish times God was loved by men in quiet, an


## 5. Train / Val Split

In [7]:
# Split 95% train / 5% val
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split_dataset["train"]
val_ds = split_dataset["test"]

print(f"Train examples: {len(train_ds):,}")
print(f"Val examples: {len(val_ds):,}")

Train examples: 2,647
Val examples: 140


## 6. Save Datasets to Disk

In [6]:
train_path = os.path.join(OUTPUT_DIR, "qa_dataset_train")
val_path = os.path.join(OUTPUT_DIR, "qa_dataset_val")

train_ds.save_to_disk(train_path)
val_ds.save_to_disk(val_path)

print(f"Saved train dataset to: {train_path}")
print(f"Saved val dataset to: {val_path}")

Saving the dataset (0/1 shards):   0%|          | 0/2647 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/140 [00:00<?, ? examples/s]

Saved train dataset to: /kaggle/working/qa_dataset_train
Saved val dataset to: /kaggle/working/qa_dataset_val
